In [1]:
import pandas as pd
import sqlite3
import os

Data loading

In [ ]:
base_path = "f:/"  
years = ["2023", "2024"]

In [3]:
# Connect to SQL

conn = sqlite3.connect("LOT_benchmark_1.db")

In [6]:
# Import all CSV

for year in years:
    year_path = os.path.join(base_path, year)
    files = [f for f in os.listdir(year_path) if f.endswith(".csv")]

    for file in files:
        table_name = file.replace(".csv", "")
        file_path = os.path.join(year_path, file)

        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)


# Check tables in database

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("\nDatabase Tables:")
print(tables)


Database Tables:
          name
0   2023_LOT_A
1   2023_LOT_B
2   2023_LOT_C
3   2023_LOT_D
4   2023_LOT_E
5   2023_LOT_F
6   2023_LOT_G
7   2023_LOT_H
8   2023_LOT_I
9   2023_LOT_J
10  2024_LOT_A
11  2024_LOT_B
12  2024_LOT_C
13  2024_LOT_D
14  2024_LOT_E
15  2024_LOT_F
16  2024_LOT_G
17  2024_LOT_H
18  2024_LOT_I
19  2024_LOT_J


Data processing

In [100]:
# Check NaN value

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
table_names = tables['name'].tolist()
print(table_names)


for table in table_names:
    print(f"\n Missing values in {table} ...")
    df = pd.read_sql(f"SELECT * FROM '{table}'", conn)
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("No missing values")
    else:
        print(missing)

['2023_LOT_A', '2023_LOT_B', '2023_LOT_C', '2023_LOT_D', '2023_LOT_E', '2023_LOT_F', '2023_LOT_G', '2023_LOT_H', '2023_LOT_I', '2023_LOT_J', '2024_LOT_A', '2024_LOT_B', '2024_LOT_C', '2024_LOT_D', '2024_LOT_E', '2024_LOT_F', '2024_LOT_G', '2024_LOT_H', '2024_LOT_I', '2024_LOT_J']

 Missing values in 2023_LOT_A ...
No missing values

 Missing values in 2023_LOT_B ...
No missing values

 Missing values in 2023_LOT_C ...
No missing values

 Missing values in 2023_LOT_D ...
Country     7724
Geo_code    7724
dtype: int64

 Missing values in 2023_LOT_E ...
Profile_label_-_category        4329
Profile_label_-_sub-category    4329
Code                            4329
dtype: int64

 Missing values in 2023_LOT_F ...
No missing values

 Missing values in 2023_LOT_G ...
No missing values

 Missing values in 2023_LOT_H ...
No missing values

 Missing values in 2023_LOT_I ...
No missing values

 Missing values in 2023_LOT_J ...
Eng._Rate       90
Lien_image     434
Lien_vidéo    1385
dtype: int64

 

In [7]:
# Modify columns' names

for table in tables["name"]:
    df = pd.read_sql(f"SELECT * FROM '{table}'", conn)
    df.columns = [col.replace(" ", "_") for col in df.columns]
    df.to_sql(table, conn, if_exists="replace", index=False)

Analysis

1. Brand Ranking 2023 vs 2024 by  
EMV /Impressions /Engagements /Post /Profiles

In [44]:
# Calculate brand ranking by KPIs and add market

def brand_ranking(year):
    query = f"""
    SELECT
        I.Brand_name AS brand_name,
        D.Geo_code AS market,
        SUM(J.Emv) AS total_emv,
        SUM(J.Impressions) AS total_impressions,
        SUM(J.Likes + J.Comments + J.Shares) AS total_engagements,
        COUNT(DISTINCT J.Post_id) AS total_posts,
        COUNT(DISTINCT C.Ally_id) AS total_profiles
    FROM "{year}_LOT_J" AS J
    LEFT JOIN "{year}_LOT_H" AS H ON J.Post_id = H.Post_id
    LEFT JOIN "{year}_LOT_I" AS I ON H.Brand_id = I.Brand_id
    LEFT JOIN "{year}_LOT_A" AS A ON H.Post_id = A.Post_id
    LEFT JOIN "{year}_LOT_C" AS C ON A.Account_id = C.Account_id
    LEFT JOIN (
        SELECT DISTINCT Ally_id, Geo_code
        FROM "{year}_LOT_D"
    ) AS D ON C.Ally_id = D.Ally_id
    GROUP BY I.Brand_name, D.Geo_code
    ORDER BY total_emv DESC;
    """
    df = pd.read_sql(query, conn)

    df["rank_emv"] = df["total_emv"].rank(ascending=False, method="dense")
    df["rank_impressions"] = df["total_impressions"].rank(ascending=False, method="dense")
    df["rank_engagements"] = df["total_engagements"].rank(ascending=False, method="dense")
    df["rank_posts"] = df["total_posts"].rank(ascending=False, method="dense")
    df["rank_profiles"] = df["total_profiles"].rank(ascending=False, method="dense")

    df["year"] = year
    return df

In [45]:
rank_2023 = brand_ranking("2023")
rank_2024 = brand_ranking("2024")

brand_compare = rank_2023.merge(
    rank_2024,
    on=["brand_name", "market"],   
    suffixes=("_2023", "_2024"),
    how="outer"  
)

In [46]:
print(brand_compare)

           brand_name market  total_emv_2023  total_impressions_2023  \
0                None   None      26348260.0            1.756907e+09   
1          YSL Beauty   None      17489169.0            1.166063e+09   
2       MAC Cosmetics   None      17314033.0            1.154402e+09   
3             Lancome   None      13885839.0            9.258288e+08   
4       Armani Beauty   None      11755431.0            7.837454e+08   
...               ...    ...             ...                     ...   
3802     Gucci Beauty     CV             NaN                     NaN   
3803           La Mer     CV             NaN                     NaN   
3804  Tom Ford Beauty     CV             NaN                     NaN   
3805           Byredo     PS             NaN                     NaN   
3806          Benefit     ET             NaN                     NaN   

      total_engagements_2023  total_posts_2023  total_profiles_2023  \
0                 98552230.0           12371.0                  

In [47]:
# Calculate all region

all_markets = (
    brand_compare.groupby("brand_name", as_index=False)[
        ["total_emv_2023", "total_emv_2024",
         "total_impressions_2023", "total_impressions_2024",
         "total_engagements_2023", "total_engagements_2024",
         "total_posts_2023", "total_posts_2024",
         "total_profiles_2023", "total_profiles_2024"]
    ].sum()
)

all_markets["market"] = "All Markets"

brand_compare_with_all = pd.concat([brand_compare, all_markets], ignore_index=True)
print(brand_compare_with_all)

              brand_name       market  total_emv_2023  total_impressions_2023  \
0                   None         None      26348260.0            1.756907e+09   
1             YSL Beauty         None      17489169.0            1.166063e+09   
2          MAC Cosmetics         None      17314033.0            1.154402e+09   
3                Lancome         None      13885839.0            9.258288e+08   
4          Armani Beauty         None      11755431.0            7.837454e+08   
...                  ...          ...             ...                     ...   
3855         Urban Decay  All Markets      11918278.0            7.948106e+08   
3856    Valentino Beauty  All Markets       1498006.0            9.988392e+07   
3857  Versace Fragrances  All Markets         28988.0            1.934873e+06   
3858          YSL Beauty  All Markets      36126569.0            2.408716e+09   
3859        bareMinerals  All Markets       1453750.0            9.693670e+07   

      total_engagements_202

2. Year over year evolution of different KPIs

In [19]:
import numpy as np

In [48]:
# Calculate YOY evolution

for metric in ["emv", "impressions", "engagements", "posts", "profiles"]:
    c2023 = f"total_{metric}_2023"
    c2024 = f"total_{metric}_2024"
    brand_compare_with_all[f"{metric}_yoy_change"] = (
        (brand_compare_with_all[c2024] - brand_compare_with_all[c2023]) /
        brand_compare_with_all[c2023].replace({0: pd.NA})
    )

brand_compare_with_all = brand_compare_with_all.sort_values(
    by="emv_yoy_change", 
    ascending=False
).reset_index(drop=True)

In [49]:
print(brand_compare_with_all)

           brand_name market  total_emv_2023  total_impressions_2023  \
0       Armani Beauty     IN            71.0                  4880.0   
1           Too Faced     GR            13.0                   925.0   
2       MAC Cosmetics     SG            21.0                  1458.0   
3           Anastasia     TN             9.0                   666.0   
4       MAC Cosmetics     IR             1.0                    84.0   
...               ...    ...             ...                     ...   
3855     Gucci Beauty     CV             NaN                     NaN   
3856           La Mer     CV             NaN                     NaN   
3857  Tom Ford Beauty     CV             NaN                     NaN   
3858           Byredo     PS             NaN                     NaN   
3859          Benefit     ET             NaN                     NaN   

      total_engagements_2023  total_posts_2023  total_profiles_2023  \
0                      148.0               4.0                  

In [51]:
ami_rows = brand_compare_with_all[brand_compare_with_all["brand_name"] == "Armani Beauty"]
ami_rows.head(10)

,brand_name,market,total_emv_2023,total_impressions_2023,total_engagements_2023,total_posts_2023,total_profiles_2023,rank_emv_2023,rank_impressions_2023,rank_engagements_2023,...,rank_impressions_2024,rank_engagements_2024,rank_posts_2024,rank_profiles_2024,year_2024,emv_yoy_change,impressions_yoy_change,engagements_yoy_change,posts_yoy_change,profiles_yoy_change
0,Armani Beauty,IN,71.0,4880.0,148.0,4.0,4.0,1466.0,1897.0,1748.0,...,141.0,172.0,235.0,199.0,2024,7607.408451,7379.225615,8261.743243,20.250000,7.25
36,Armani Beauty,SR,41.0,2761.0,173.0,1.0,1.0,1496.0,2043.0,1727.0,...,557.0,1687.0,315.0,231.0,2024,586.000000,580.160811,8.289017,3.000000,0.0
97,Armani Beauty,PR,538.0,36062.0,980.0,5.0,3.0,1147.0,1221.0,1330.0,...,323.0,643.0,289.0,221.0,2024,187.622677,186.630082,59.472449,5.000000,2.666667
105,Armani Beauty,HN,80.0,5426.0,330.0,2.0,2.0,1458.0,1864.0,1610.0,...,700.0,523.0,308.0,229.0,2024,162.212500,159.472171,299.615152,4.500000,0.5
109,Armani Beauty,CA,218.0,14671.0,1066.0,6.0,4.0,1333.0,1527.0,1302.0,...,478.0,396.0,283.0,217.0,2024,158.903670,157.474542,184.212946,5.000000,2.75
121,Armani Beauty,PH,11.0,797.0,29.0,1.0,1.0,1526.0,2300.0,1862.0,...,1339.0,1310.0,278.0,221.0,2024,135.272727,125.962359,182.103448,40.000000,10.0
145,Armani Beauty,TR,40.0,2715.0,140.0,2.0,2.0,1497.0,2050.0,1755.0,...,988.0,979.0,292.0,224.0,2024,109.400000,107.709392,105.900000,12.500000,3.0
149,Armani Beauty,CO,38.0,2564.0,92.0,1.0,1.0,1499.0,2066.0,1800.0,...,1015.0,843.0,298.0,223.0,2024,105.315789,104.271841,263.706522,20.000000,8.0
165,Armani Beauty,MY,1058.0,70666.0,1688.0,7.0,6.0,978.0,1005.0,1176.0,...,331.0,837.0,317.0,231.0,2024,89.249527,89.080208,14.040877,-0.714286,-0.833333
178,Armani Beauty,NG,1616.0,107985.0,7944.0,6.0,5.0,868.0,876.0,772.0,...,287.0,306.0,285.0,220.0,2024,79.914604,79.733741,43.270015,4.666667,1.4


3. Market share of Posts, Profiles and Impressions

In [52]:
# Calculate market share in 2023 & 2024

metrics = ["posts", "profiles", "impressions"]

for metric in metrics:
    # 2023
    total_2023 = brand_compare_with_all.groupby("market")[f"total_{metric}_2023"].transform("sum")
    brand_compare_with_all[f"{metric}_market_share_2023"] = (
        brand_compare_with_all[f"total_{metric}_2023"] / total_2023 * 100
    )
    
    # 2024
    total_2024 = brand_compare_with_all.groupby("market")[f"total_{metric}_2024"].transform("sum")
    brand_compare_with_all[f"{metric}_market_share_2024"] = (
        brand_compare_with_all[f"total_{metric}_2024"] / total_2024 * 100
    )



In [ ]:
# Calculate market share in 2023 & 2024

metrics = ["emv", "impressions", "engagements", "posts", "profiles"]

for metric in metrics:

    total_2023 = brand_compare_with_all.groupby("market")[f"total_{metric}_2023"].transform("sum")
    brand_compare_with_all[f"{metric}_market_share_2023"] = (
        brand_compare_with_all[f"total_{metric}_2023"] / total_2023 * 100
    )
    total_2024 = brand_compare_with_all.groupby("market")[f"total_{metric}_2024"].transform("sum")
    brand_compare_with_all[f"{metric}_market_share_2024"] = (
        brand_compare_with_all[f"total_{metric}_2024"] / total_2024 * 100
    )


for metric in metrics:
    mask = brand_compare_with_all["market"] == "All Markets"
    total_2023_all = brand_compare_with_all.loc[mask, f"total_{metric}_2023"].sum()
    total_2024_all = brand_compare_with_all.loc[mask, f"total_{metric}_2024"].sum()
    brand_compare_with_all.loc[mask, f"{metric}_market_share_2023"] = (
        brand_compare_with_all.loc[mask, f"total_{metric}_2023"] / total_2023_all * 100
    )
    brand_compare_with_all.loc[mask, f"{metric}_market_share_2024"] = (
        brand_compare_with_all.loc[mask, f"total_{metric}_2024"] / total_2024_all * 100
    )

In [85]:
# Calculate YOY evolution of market share

for metric in metrics:
    c2023 = f"{metric}_market_share_2023"
    c2024 = f"{metric}_market_share_2024"
    brand_compare_with_all[f"{metric}_market_share_change"] = (
        (brand_compare_with_all[c2024] - brand_compare_with_all[c2023]) /
        brand_compare_with_all[c2023].replace({0: pd.NA}) 
    )

brand_compare_with_all.head(10)

,brand_name,market,total_emv_2023,total_impressions_2023,total_engagements_2023,total_posts_2023,total_profiles_2023,rank_emv_2023,rank_impressions_2023,rank_engagements_2023,...,impressions_market_share_2024,posts_market_share_change,profiles_market_share_change,impressions_market_share_change,emv_market_share_2023,emv_market_share_2024,engagements_market_share_2023,engagements_market_share_2024,emv_market_share_change,engagements_market_share_change
0,Armani Beauty,IN,71.0,4880.0,148.0,4.0,4.0,1466.0,1897.0,1748.0,...,5.818310,3.616213,1.252497,445.742933,0.012637,5.818495,0.006683,3.610818,459.436367,539.301358
1,Too Faced,GR,13.0,925.0,21.0,1.0,1.0,1524.0,2273.0,1870.0,...,8.450231,1.445783,1.647059,1050.206146,0.007533,8.450808,0.005334,13.890189,1120.792765,2602.901158
2,MAC Cosmetics,SG,21.0,1458.0,69.0,1.0,1.0,1516.0,2182.0,1822.0,...,34.104033,7.876404,2.022222,2179.644870,0.015021,34.106163,0.026581,37.240535,2269.560934,1400.042891
3,Anastasia,TN,9.0,666.0,30.0,1.0,1.0,1528.0,2333.0,1861.0,...,37.902313,14.130435,4.205882,1790.304859,0.019082,37.915384,0.021474,41.366627,1985.976782,1925.374876
4,MAC Cosmetics,IR,1.0,84.0,3.0,1.0,1.0,1536.0,2434.0,1888.0,...,7.342315,-0.197183,-0.228070,131.739432,0.044072,7.353222,0.046780,8.652127,165.844611,183.953641
5,Bobbi Brown,NG,8.0,580.0,15.0,1.0,1.0,1529.0,2354.0,1876.0,...,0.791060,6.624021,2.470085,301.597277,0.002405,0.790932,0.000760,0.828143,327.840014,1088.168612
6,Dior Beauty,EG,20.0,1385.0,119.0,1.0,1.0,1517.0,2194.0,1775.0,...,9.611705,2.111111,2.207921,207.518054,0.044401,9.612777,0.066467,8.609077,215.498958,128.523919
7,The Ordinary,PS,28.0,1929.0,75.0,1.0,1.0,1509.0,2119.0,1816.0,...,40.922774,10.658363,1.153846,2905.900596,0.013627,40.946776,0.004581,24.357185,3003.762134,5315.787073
8,The Ordinary,JP,51.0,3487.0,421.0,3.0,3.0,1486.0,1979.0,1553.0,...,2.348103,3.927989,3.233766,1819.607175,0.001258,2.348133,0.006373,0.767164,1866.082116,119.383915
9,Kylie Cosmetics,ZA,71.0,4804.0,204.0,2.0,2.0,1466.0,1902.0,1700.0,...,2.915843,7.521567,4.796225,562.065678,0.005103,2.915989,0.010223,7.426932,570.394971,725.499228


In [54]:
print(brand_compare_with_all)

           brand_name market  total_emv_2023  total_impressions_2023  \
0       Armani Beauty     IN            71.0                  4880.0   
1           Too Faced     GR            13.0                   925.0   
2       MAC Cosmetics     SG            21.0                  1458.0   
3           Anastasia     TN             9.0                   666.0   
4       MAC Cosmetics     IR             1.0                    84.0   
...               ...    ...             ...                     ...   
3855     Gucci Beauty     CV             NaN                     NaN   
3856           La Mer     CV             NaN                     NaN   
3857  Tom Ford Beauty     CV             NaN                     NaN   
3858           Byredo     PS             NaN                     NaN   
3859          Benefit     ET             NaN                     NaN   

      total_engagements_2023  total_posts_2023  total_profiles_2023  \
0                      148.0               4.0                  

In [55]:
all_markets_rows = brand_compare_with_all[brand_compare_with_all["market"] == "All Markets"]
print(all_markets_rows[["brand_name", "posts_market_share_change"]])

                      brand_name  posts_market_share_change
494            Helena Rubinstein                  -0.488077
667         Dolce Gabbana Beauty                  -0.306073
682             Valentino Beauty                   1.473097
686           Versace Fragrances                  -0.148526
732                   Shu Uemura                  -0.298656
782              Givenchy Beauty                   0.292056
837            Rabanne Fragrance                   0.144144
849                 Prada beauty                   1.872865
852                Laura Mercier                   0.027705
860   Carolina Herrera Fragrance                  -0.090377
875                     Shiseido                   0.269927
889                    Anastasia                   0.580289
943                  Cle de Peau                  -0.114290
969              Tarte Cosmetics                   0.747145
988                     Clinique                  -0.067578
1008           Charlotte Tilbury        

Push table to Bigquery & Firebase

In [58]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pyarrow
import tempfile
import shutil

In [95]:
# Push

tmp_path = os.path.join(tempfile.gettempdir(), "brand_compare_KPI.csv")
brand_compare_with_all.to_csv(tmp_path, index=False, encoding="utf-8")

job = client.load_table_from_file(
    open(tmp_path, "rb"),
    table_ref,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True
    )
)

job.result()


LoadJob<project=benchmarkcasestudy, location=US, id=c2d82abc-9104-4de5-bf58-2071e42d84e7>

In [72]:
import firebase_admin
from firebase_admin import credentials, firestore

In [63]:
db = firestore.client()
collection_name = "brand_compare_KPI"

In [64]:
# Accelerate

batch = db.batch()
batch_size = 500 
counter = 0

4. Post Monthly Evolution

In [27]:
# Define function to calculate post monthly evolution

def post_monthly_evolution(year):
    query = f"""
    SELECT
        J.Timestamp AS date,
        I.Brand_name AS brand_name,
        J.Emv AS emv,
        J.Impressions AS impressions,
        J."Eng._Rate" AS engagement_rate,
        J.Followers AS followers,
        J.Post_id AS posts
    FROM "{year}_LOT_J" AS J
    LEFT JOIN "{year}_LOT_H" AS H ON J.Post_id = H.Post_id
    LEFT JOIN "{year}_LOT_I" AS I ON H.Brand_id = I.Brand_id
    WHERE J."Timestamp" IS NOT NULL
    """
    
    df = pd.read_sql(query, conn)

    # Convert date format and extract month

    df["date"] = pd.to_datetime(df["date"], errors="coerce", infer_datetime_format=True)
    df["month"] = df["date"].dt.to_period("M").astype(str)

    # Calculate engagement 
    
    if "engagement_rate" in df.columns:
        df["engagements"] = df["followers"] * df["engagement_rate"]
    else:
        df["engagements"] = None

    # Group by brand name

    monthly = df.groupby(["month", "brand_name"], as_index=False).agg(
        total_posts=("posts", "nunique"),
        total_impressions=("impressions", "sum"),
        total_engagements=("engagements", "sum"),
        total_emv=("emv", "sum")
    )

    monthly["year"] = year

    return monthly

In [28]:
monthly_2023 = post_monthly_evolution("2023")

C:\Users\z\AppData\Local\Temp\ipykernel_2784\615885820.py:23: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["date"] = pd.to_datetime(df["date"], errors="coerce", infer_datetime_format=True)


In [29]:
print(monthly_2023)

       month          brand_name  total_posts  total_impressions  \
0    2023-01           Anastasia          694         45647171.0   
1    2023-01       Armani Beauty          138        128725841.0   
2    2023-01             Benefit          451        139708366.0   
3    2023-01            Biotherm            5           159504.0   
4    2023-01         Bobbi Brown          160         30874235.0   
..       ...                 ...          ...                ...   
625  2023-12         Urban Decay          650         84457487.0   
626  2023-12    Valentino Beauty           98          7599674.0   
627  2023-12  Versace Fragrances           13           134100.0   
628  2023-12          YSL Beauty         1356        151150536.0   
629  2023-12        bareMinerals           51          4994397.0   

     total_engagements  total_emv  year  
0         2.565914e+06   684408.0  2023  
1         1.309523e+06  1930840.0  2023  
2         3.562579e+06  2095452.0  2023  
3         7.314

In [30]:
monthly_2024 = post_monthly_evolution("2024")

post_monthly = pd.concat([monthly_2023, monthly_2024]).reset_index(drop=True)
print(post_monthly)

C:\Users\z\AppData\Local\Temp\ipykernel_2784\615885820.py:23: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["date"] = pd.to_datetime(df["date"], errors="coerce", infer_datetime_format=True)


        month          brand_name  total_posts  total_impressions  \
0     2023-01           Anastasia          694         45647171.0   
1     2023-01       Armani Beauty          138        128725841.0   
2     2023-01             Benefit          451        139708366.0   
3     2023-01            Biotherm            5           159504.0   
4     2023-01         Bobbi Brown          160         30874235.0   
...       ...                 ...          ...                ...   
1254  2024-12         Urban Decay         1320        165686214.0   
1255  2024-12    Valentino Beauty          657         76232993.0   
1256  2024-12  Versace Fragrances            8            98200.0   
1257  2024-12          YSL Beauty         2649        273110760.0   
1258  2024-12        bareMinerals          184         18575364.0   

      total_engagements  total_emv  year  
0          2.565914e+06   684408.0  2023  
1          1.309523e+06  1930840.0  2023  
2          3.562579e+06  2095452.0  2023  

5. Ranking Post by Profile Category

In [32]:
# Define function to rank post by profile category

def ranking_post_by_category(year):
    query = f"""
    SELECT 
        E."Profile_label_-_category" AS profile_category,
        COUNT(DISTINCT J.Post_id) AS total_posts,
        SUM(J.Impressions) AS total_impressions,
        SUM(J.Emv) AS total_emv,
        SUM(J.Likes + J.Comments + J.Shares) AS total_engagements
    FROM "{year}_LOT_J" AS J
    LEFT JOIN (
        SELECT DISTINCT Account_id, Post_id
        FROM "{year}_LOT_A"
    ) AS A ON J.Post_id = A.Post_id
    INNER JOIN "{year}_LOT_C" AS C ON A.Account_id = C.Account_id
    LEFT JOIN "{year}_LOT_E" AS E ON C.Ally_id = E.Ally_id
    WHERE E."Profile_label_-_category" IS NOT NULL
    GROUP BY E."Profile_label_-_category"
    ORDER BY total_posts DESC;
    """

    df = pd.read_sql(query, conn)
    df["year"] = year
    return df

In [22]:
pd.read_sql('PRAGMA table_info("2023_LOT_E");', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Ally_id,TEXT,0,None,0
1,1,Username,TEXT,0,None,0
2,2,Profile_label_-_category,TEXT,0,None,0
3,3,Profile_label_-_sub-category,TEXT,0,None,0
4,4,Code,TEXT,0,None,0


In [34]:
rank_profile_2023 = ranking_post_by_category("2023")
rank_profile_2024 = ranking_post_by_category("2024")

# Merge two years

rank_profile_compare = rank_profile_2023.merge(
    rank_profile_2024, 
    on="profile_category", 
    suffixes=("_2023", "_2024"),
    how="outer"
)


print(rank_profile_compare)


  profile_category  total_posts_2023  total_impressions_2023  total_emv_2023  \
0  Digital Creator             79848            1.316793e+10     197484469.0   
1    Beauty Expert             31117            4.145768e+09      62173131.0   
2           Others              8669            1.035951e+09      15535427.0   
3        Celebrity              6427            2.636047e+09      39538155.0   
4              KOL              3301            8.234935e+08      12351065.0   
5         Business              2409            3.035719e+08       4552527.0   
6            Media               963            1.661223e+08       2491423.0   

   total_engagements_2023 year_2023  total_posts_2024  total_impressions_2024  \
0             552079928.0      2023            206341            2.171086e+10   
1             179329528.0      2023             61509            6.025056e+09   
2              32197483.0      2023             13906            1.476309e+09   
3              80146342.0      2023

In [35]:
# Calculate YOY

for metric in ["total_posts", "total_impressions", "total_emv", "total_engagements"]:
    rank_profile_compare[f"{metric}_yoy_change"] = (
        (rank_profile_compare[f"{metric}_2024"] - rank_profile_compare[f"{metric}_2023"])
        / rank_profile_compare[f"{metric}_2023"]
    )

print(rank_profile_compare)

  profile_category  total_posts_2023  total_impressions_2023  total_emv_2023  \
0  Digital Creator             79848            1.316793e+10     197484469.0   
1    Beauty Expert             31117            4.145768e+09      62173131.0   
2           Others              8669            1.035951e+09      15535427.0   
3        Celebrity              6427            2.636047e+09      39538155.0   
4              KOL              3301            8.234935e+08      12351065.0   
5         Business              2409            3.035719e+08       4552527.0   
6            Media               963            1.661223e+08       2491423.0   

   total_engagements_2023 year_2023  total_posts_2024  total_impressions_2024  \
0             552079928.0      2023            206341            2.171086e+10   
1             179329528.0      2023             61509            6.025056e+09   
2              32197483.0      2023             13906            1.476309e+09   
3              80146342.0      2023

6.  Ranking most impactful profiles per brand 

In [96]:

def impactful_profiles_2024(top_n_brands=4):
    #  Find out most visible brands（order by impressions）

    top_brands_query = """
        SELECT 
            I.Brand_name,
            SUM(J.Impressions) AS total_impressions
        FROM "2024_LOT_J" AS J
        INNER JOIN "2024_LOT_H" AS H ON J.Post_id = H.Post_id
        LEFT JOIN "2024_LOT_I" AS I ON H.Brand_id = I.Brand_id
        GROUP BY I.Brand_name
        ORDER BY total_impressions DESC
        LIMIT ?;
    """
    top_brands = pd.read_sql(top_brands_query, conn, params=(top_n_brands,))
    brand_list = tuple(top_brands["Brand_name"].tolist())

    # Split tier

    conn.execute("DROP TABLE IF EXISTS tmp_2024_tier;")
    tier_sql = """
        CREATE TEMP TABLE tmp_2024_tier AS
        SELECT 
            C.Ally_id,
            MAX(B.Followers) AS max_followers,
            CASE
                WHEN MAX(B.Followers) >= 1000000 THEN 'Mega'
                WHEN MAX(B.Followers) >= 500000 THEN 'Macro'
                WHEN MAX(B.Followers) >= 100000 THEN 'Midtier'
                WHEN MAX(B.Followers) >= 10000 THEN 'Micro'
                ELSE 'Nano'
            END AS ally_tier
        FROM "2024_LOT_B" AS B
        LEFT JOIN "2024_LOT_C" AS C ON B.Account_id = C.Account_id
        GROUP BY C.Ally_id;
    """
    conn.execute(tier_sql)
    conn.commit()


    # Profile performance

    main_query = f"""
        SELECT 
            I.Brand_name AS brand_name,
            D.Geo_code AS market,
            C.Ally_id,
            E."Profile_label_-_category" AS ally_category,
            T.ally_tier,
            STRFTIME('%Y-%m', J.Timestamp) AS month,
            SUM(J.Impressions) AS total_impressions,
            SUM(J.Emv) AS total_emv,
            SUM(J.Likes + J.Comments + J.Shares) AS total_engagements,
            MAX(T.max_followers) AS followers,
            COUNT(DISTINCT J.Post_id) AS total_posts
        FROM "2024_LOT_J" AS J
        LEFT JOIN "2024_LOT_H" AS H ON J.Post_id = H.Post_id
        LEFT JOIN "2024_LOT_I" AS I ON H.Brand_id = I.Brand_id
        LEFT JOIN "2024_LOT_A" AS A ON J.Post_id = A.Post_id
        LEFT JOIN "2024_LOT_C" AS C ON A.Account_id = C.Account_id
        LEFT JOIN "2024_LOT_D" AS D ON C.Ally_id = D.Ally_id
        LEFT JOIN "2024_LOT_E" AS E ON C.Ally_id = E.Ally_id
        LEFT JOIN tmp_2024_tier AS T ON C.Ally_id = T.Ally_id
        WHERE I.Brand_name IN {brand_list}
        GROUP BY 
           month,I.Brand_name, D.Geo_code,
           C.Ally_id, E."Profile_label_-_category", ally_tier
        ORDER BY total_emv DESC;
    """

    df = pd.read_sql(main_query, conn)
    df["year"] = 2024

    all_markets = (
        df.groupby(
            ["brand_name", "Ally_id", "ally_category", "ally_tier", "month"],
            as_index=False
        )[["total_impressions", "total_emv", "total_engagements", "followers", "total_posts"]]
        .sum()
    )
    all_markets["market"] = "All Markets"

    df_with_all = pd.concat([df, all_markets], ignore_index=True)

    return df_with_all



In [97]:
impactful_2024 = impactful_profiles_2024(top_n_brands=4)
print(impactful_2024)

           brand_name       market                      Ally_id  \
0       Armani Beauty         None                  ig:43678222   
1             Lancome           MX       tk:6788899448527504389   
2          YSL Beauty         None                 ig:304358009   
3          YSL Beauty         None               ig:56105303581   
4       Armani Beauty         None               ig:49734649834   
...               ...          ...                          ...   
122894     YSL Beauty  All Markets  yt:UCxbo-dmcf8PX1uhPLU76YhA   
122895     YSL Beauty  All Markets  yt:UCxbo-dmcf8PX1uhPLU76YhA   
122896     YSL Beauty  All Markets  yt:UCxbo-dmcf8PX1uhPLU76YhA   
122897     YSL Beauty  All Markets  yt:UCxbo-dmcf8PX1uhPLU76YhA   
122898     YSL Beauty  All Markets  yt:UCyujzJvIm9hSBi6qiyT18BQ   

          ally_category ally_tier    month  total_impressions  total_emv  \
0             Celebrity      Mega  2024-11        124400000.0  1866000.0   
1       Digital Creator     Macro  2024-09 